In [1]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path

# new import
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,name"

# ======================
# Extracción
# ======================
@task
def extract_names():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

# ======================
# Transformación
# ======================
@task
def transform_names(data):
    df = pd.DataFrame(data)

    # Tabla principal: nombres comunes y oficiales
    df_main = df[["cca3"]].copy()
    df_main["common_name"] = df["name"].apply(lambda x: x.get("common") if isinstance(x, dict) else None)
    df_main["official_name"] = df["name"].apply(lambda x: x.get("official") if isinstance(x, dict) else None)
    df_main = df_main.rename(columns={"cca3": "id_country"})

    # Tabla de nombres nativos
    df["native_list"] = df["name"].apply(
        lambda x: list(x.get("nativeName", {}).items()) if isinstance(x, dict) and "nativeName" in x else []
    )
    df_native = df[["cca3", "native_list"]].explode("native_list").dropna(subset=["native_list"])

    if not df_native.empty:
        df_native[["id_language", "values"]] = pd.DataFrame(df_native["native_list"].tolist(), index=df_native.index)
        df_native["native_official"] = df_native["values"].apply(lambda x: x.get("official"))
        df_native["native_common"] = df_native["values"].apply(lambda x: x.get("common"))
        df_native = df_native.rename(columns={"cca3": "id_country"})
        df_native = df_native[["id_country", "id_language", "native_common", "native_official"]]
    else:
        df_native = pd.DataFrame(columns=["id_country", "id_language", "native_common", "native_official"])

    return df_main, df_native

# ======================
# Validación
# ======================
country_names_schema = DataFrameSchema({
    "id_country": Column(str, checks=Check.str_length(3, 3), nullable=False),
    "common_name": Column(str, nullable=True),
    "official_name": Column(str, nullable=True),
})

country_native_schema = DataFrameSchema({
    "id_country": Column(str, checks=Check.str_length(3, 3), nullable=False),
    "id_language": Column(str, checks=Check.str_length(2, 10), nullable=False),
    "native_common": Column(str, nullable=True),
    "native_official": Column(str, nullable=True),
})

@task
def validate_names(df_main: pd.DataFrame, df_native: pd.DataFrame):
    df_main = country_names_schema.validate(df_main)
    df_native = country_native_schema.validate(df_native)
    return df_main, df_native

# ======================
# Carga
# ======================
@task
def load_names(df_main: pd.DataFrame, df_native: pd.DataFrame):
    stage_dir = Path("../stage")
    stage_dir.mkdir(parents=True, exist_ok=True)

    file_main = stage_dir / "country_names.csv"
    file_native = stage_dir / "country_native_names.csv"

    df_main.to_csv(file_main, index=False, encoding="utf-8")
    df_native.to_csv(file_native, index=False, encoding="utf-8")

    print(f"Saved {len(df_main)} rows to {file_main}")
    print(f"Saved {len(df_native)} rows to {file_native}")

# ======================
# Flujo
# ======================
@flow(name="etl_country_names_flow")
def etl_country_names():
    data = extract_names()
    df_main, df_native = transform_names(data)
    df_main, df_native = validate_names(df_main, df_native)
    load_names(df_main, df_native)

if __name__ == "__main__":
    etl_country_names()


C:\Users\Sebastian\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandera\_pandas_deprecated.py:149: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


04:46:51.003 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8049
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

04:46:55.402 | INFO    | Flow run 'antique-bison' - Beginning flow run 'antique-bison' for flow 'etl_country_names_flow'

04:46:56.781 | INFO    | Task run 'extract_names-36a' - Finished in state Completed()

04:46:57.049 | INFO    | Task run 'transform_names-231' - Finished in state Completed()

04:46:57.288 | INFO    | Task run 'validate_names-320' - Finished in state Completed()

Saved 250 rows to ..\stage\country_names.csv
Saved 410 rows to ..\stage\country_native_names.csv


04:46:57.513 | INFO    | Task run 'load_names-3be' - Finished in state Completed()

04:46:57.538 | INFO    | Flow run 'antique-bison' - Finished in state Completed()